# Character-level representation cho NER trên noisy text (WNUT 17)

**Câu hỏi nghiên cứu:** Character representation có giúp mô hình nhận diện entity unseen và rare tốt hơn không?

Notebook dùng dữ liệu `dany0407/wnut_17`, PyTorch và các mô hình huấn luyện hoàn toàn từ đầu (không pretrained embeddings/model). So sánh: (1) word-BiLSTM-Softmax, (2) word-BiLSTM-CRF, (3) word+character-CNN-BiLSTM-CRF. Mô hình (2) đồng thời là ablation tắt character của (3), giữ nguyên word encoder và CRF.

Entity **unseen** có surface form chưa xuất hiện trong entity train; **rare** xuất hiện 1–2 lần trong train; **frequent** xuất hiện ít nhất 3 lần. Tần suất chỉ được tính từ train. Báo cáo entity-level exact-span micro-F1 và bootstrap theo câu trên test để so sánh mô hình character với ablation. Bootstrap phản ánh độ bất định do lấy mẫu test, không thay thế nhiều lần chạy với seed khác nhau.

Nếu chưa cài package, chạy cell setup. Nếu máy không có CUDA, notebook vẫn chạy CPU; giảm `EPOCHS`/`BATCH_SIZE` để thử nhanh. Thời gian tùy cấu hình máy.

In [1]:
# Cài nếu thiếu: %pip install datasets pandas matplotlib scikit-learn
import os, random, re, time, collections, math
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.set_num_threads(min(4, os.cpu_count() or 1))
# Giữ cấu hình gọn để notebook có thể chạy cả trên CPU.
EPOCHS = 12
BATCH_SIZE = 32
WORD_DIM = 100
HIDDEN_DIM = 64
PATIENCE = 4
print(f'PyTorch {torch.__version__} | device={DEVICE}')
if DEVICE.type == 'cuda': print(torch.cuda.get_device_name(0))

PyTorch 2.15.0+cpu | device=cpu


## 1. Nạp và kiểm tra dữ liệu

Ưu tiên validation/test có sẵn; chỉ tách validation từ train theo seed nếu dataset không cung cấp. Giữ nguyên tokenization và tag sequence của nguồn dữ liệu.

In [2]:
ds = load_dataset('dany0407/wnut_17')
print(ds)
print('Columns:', ds[next(iter(ds.keys()))].column_names)
print('Features:', ds[next(iter(ds.keys()))].features)
train_raw = ds['train']
test_raw = ds['test']
if 'validation' in ds:
    valid_raw = ds['validation']
elif 'dev' in ds:
    valid_raw = ds['dev']
else:
    split = train_raw.train_test_split(test_size=0.1, seed=SEED)
    train_raw, valid_raw = split['train'], split['test']

def find_column(columns, candidates):
    for name in candidates:
        if name in columns: return name
    raise KeyError(f'Không tìm thấy cột trong {columns}; cần một trong {candidates}')

token_col = find_column(train_raw.column_names, ['tokens', 'words', 'text'])
tag_col = find_column(train_raw.column_names, ['ner_tags', 'tags', 'labels'])
tag_feature = train_raw.features[tag_col]
tag_names = getattr(getattr(tag_feature, 'feature', tag_feature), 'names', None)
if tag_names is None:
    raise ValueError('Không đọc được danh sách nhãn từ feature; kiểm tra schema của dataset.')
print(f'Splits: train={len(train_raw)}, validation={len(valid_raw)}, test={len(test_raw)}')
print('Tags:', tag_names)
print('Example:', train_raw[0][token_col][:12], train_raw[0][tag_col][:12])

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'ner_tags'],
        num_rows: 3394
    })
    validation: Dataset({
        features: ['id', 'tokens', 'ner_tags'],
        num_rows: 1009
    })
    test: Dataset({
        features: ['id', 'tokens', 'ner_tags'],
        num_rows: 1287
    })
})
Columns: ['id', 'tokens', 'ner_tags']
Features: {'id': Value('string'), 'tokens': List(Value('string')), 'ner_tags': List(ClassLabel(names=['O', 'B-corporation', 'I-corporation', 'B-creative-work', 'I-creative-work', 'B-group', 'I-group', 'B-location', 'I-location', 'B-person', 'I-person', 'B-product', 'I-product']))}
Splits: train=3394, validation=1009, test=1287
Tags: ['O', 'B-corporation', 'I-corporation', 'B-creative-work', 'I-creative-work', 'B-group', 'I-group', 'B-location', 'I-location', 'B-person', 'I-person', 'B-product', 'I-product']
Example: ['@paulwalk', 'It', "'s", 'the', 'view', 'from', 'where', 'I', "'m", 'living', 'for', 'two'] [0, 0, 0, 0, 0, 0, 0, 0, 0, 0

## 2. Chuẩn hóa nhãn và định nghĩa unseen/rare

Entity surface là chuỗi token ghép bằng khoảng trắng, chuẩn hóa lowercase chỉ để đếm tần suất. Nhãn BIO được chuyển thành span; parser chịu được tiền tố I không hợp lệ bằng cách bắt đầu span mới. Báo cáo riêng span không có trong train và span có tần suất 1–2.

In [3]:
def normalize_split(split):
    rows = []
    for row in split:
        tokens = [str(x) for x in row[token_col]]
        raw_tags = row[tag_col]
        tags = [tag_names[int(x)] if not isinstance(x, str) else x for x in raw_tags]
        if len(tokens) != len(tags):
            raise ValueError(f'Token/tag length mismatch: {len(tokens)} != {len(tags)}')
        rows.append({'tokens': tokens, 'tags': tags})
    return rows

train = normalize_split(train_raw)
valid = normalize_split(valid_raw)
test = normalize_split(test_raw)
tag_set = sorted({tag for row in train for tag in row['tags']})
tag2id = {tag: i for i, tag in enumerate(tag_set)}
id2tag = {i: tag for tag, i in tag2id.items()}
if 'O' not in tag2id: raise ValueError('Nhãn O không có trong tập train.')

def parse_spans(tags):
    spans, start, active_type = [], None, None
    def close(end):
        nonlocal start, active_type
        if start is not None: spans.append((start, end, active_type))
        start, active_type = None, None
    for i, tag in enumerate(list(tags) + ['O']):
        if tag == 'O' or tag == '':
            close(i); continue
        if '-' in tag:
            prefix, entity_type = tag.split('-', 1)
        else:
            prefix, entity_type = 'B', tag
        if prefix in ('B', 'S', 'U'):
            close(i)
            if prefix in ('S', 'U'):
                spans.append((i, i + 1, entity_type))
            else:
                start, active_type = i, entity_type
        elif prefix in ('I', 'E', 'L'):
            if start is None or active_type != entity_type:
                close(i); start, active_type = i, entity_type
            if prefix in ('E', 'L'):
                close(i + 1)
        else:
            close(i); start, active_type = i, entity_type
    return spans

def surface(tokens, span):
    a, b, _ = span
    return ' '.join(tokens[a:b])

entity_freq = collections.Counter(
    surface(row['tokens'], span).casefold()
    for row in train for span in parse_spans(row['tags'])
)
def entity_group(tokens, span):
    count = entity_freq[surface(tokens, span).casefold()]
    return 'unseen' if count == 0 else ('rare (1-2)' if count <= 2 else 'frequent (3+)')

print('Tag vocabulary:', tag2id)
print('Entity occurrences in train:', sum(entity_freq.values()))
group_counts = collections.Counter(entity_group(row['tokens'], s) for row in test for s in parse_spans(row['tags']))
print('Gold test entity counts:', dict(group_counts))

Tag vocabulary: {'B-corporation': 0, 'B-creative-work': 1, 'B-group': 2, 'B-location': 3, 'B-person': 4, 'B-product': 5, 'I-corporation': 6, 'I-creative-work': 7, 'I-group': 8, 'I-location': 9, 'I-person': 10, 'I-product': 11, 'O': 12}
Entity occurrences in train: 1975
Gold test entity counts: {'unseen': 1055, 'rare (1-2)': 22, 'frequent (3+)': 2}


## 3. Vocabulary, batching và character encoder

Word vocabulary và character vocabulary chỉ học từ train; word OOV dùng `<UNK>`. Character CNN mã hóa từng token, ghép vector ký tự với word embedding rồi đưa vào BiLSTM.

In [4]:
PAD_WORD, UNK_WORD, PAD_CHAR, UNK_CHAR = '<PAD>', '<UNK>', '<PAD_CHAR>', '<UNK_CHAR>'
word_counts = collections.Counter(tok for row in train for tok in row['tokens'])
word2id = {PAD_WORD: 0, UNK_WORD: 1}
for word in word_counts: word2id.setdefault(word, len(word2id))
char2id = {PAD_CHAR: 0, UNK_CHAR: 1}
for row in train:
    for token in row['tokens']:
        for char in token: char2id.setdefault(char, len(char2id))
PAD_TAG = -100
tag_counts = torch.bincount(torch.tensor([tag2id[t] for row in train for t in row['tags']]), minlength=len(tag2id)).float()
# Giảm thiên lệch về nhãn O trong dữ liệu NER thưa; căn bậc hai tránh trọng số quá lớn cho nhãn hiếm.
tag_weights = (tag_counts.sum() / tag_counts.clamp_min(1)).sqrt()

class NERDataset(Dataset):
    def __init__(self, rows): self.rows = rows
    def __len__(self): return len(self.rows)
    def __getitem__(self, idx):
        row = self.rows[idx]
        words = [word2id.get(w, word2id[UNK_WORD]) for w in row['tokens']]
        tags = [tag2id[t] for t in row['tags']]
        chars = [[char2id.get(c, char2id[UNK_CHAR]) for c in w] or [char2id[UNK_CHAR]] for w in row['tokens']]
        return words, chars, tags

def collate_batch(batch):
    lengths = torch.tensor([len(x[0]) for x in batch], dtype=torch.long)
    max_tokens = int(lengths.max())
    max_chars = max(1, max(len(chars) for _, char_seq, _ in batch for chars in char_seq))
    words = torch.zeros(len(batch), max_tokens, dtype=torch.long)
    tags = torch.full((len(batch), max_tokens), PAD_TAG, dtype=torch.long)
    chars = torch.zeros(len(batch), max_tokens, max_chars, dtype=torch.long)
    char_lengths = torch.zeros(len(batch), max_tokens, dtype=torch.long)
    for i, (word_ids, char_seq, tag_ids) in enumerate(batch):
        n = len(word_ids); words[i, :n] = torch.tensor(word_ids); tags[i, :n] = torch.tensor(tag_ids)
        for j, char_ids in enumerate(char_seq):
            chars[i, j, :len(char_ids)] = torch.tensor(char_ids)
            char_lengths[i, j] = len(char_ids)
    return words, chars, char_lengths, tags, lengths

def make_loader(rows, shuffle=False):
    generator = torch.Generator().manual_seed(SEED)
    return DataLoader(NERDataset(rows), batch_size=BATCH_SIZE, shuffle=shuffle,
                      collate_fn=collate_batch, generator=generator, num_workers=0)

train_loader, valid_loader, test_loader = make_loader(train, True), make_loader(valid), make_loader(test)
print(f'word vocab={len(word2id):,}; char vocab={len(char2id):,}; labels={len(tag2id)}')
print('Train batches:', len(train_loader), '| Example batch shapes:', [tuple(x.shape) for x in next(iter(train_loader))])

word vocab=14,880; char vocab=94; labels=13
Train batches: 107 | Example batch shapes: [(32, 33), (32, 33, 66), (32, 33), (32, 33), (32,)]


## 4. Mô hình huấn luyện từ đầu

CRF tối ưu chuỗi nhãn hợp lệ trên toàn câu. Word-BiLSTM-Softmax là baseline 1; Word-BiLSTM-CRF là baseline 2 và ablation character-off; Word+Char-CNN-BiLSTM-CRF là mô hình đề xuất.

In [5]:
class LinearChainCRF(nn.Module):
    def __init__(self, num_tags):
        super().__init__()
        self.transitions = nn.Parameter(torch.empty(num_tags, num_tags))
        self.start = nn.Parameter(torch.empty(num_tags)); self.end = nn.Parameter(torch.empty(num_tags))
        nn.init.uniform_(self.transitions, -0.1, 0.1)
        nn.init.uniform_(self.start, -0.1, 0.1); nn.init.uniform_(self.end, -0.1, 0.1)

    def nll(self, emissions, tags, lengths):
        batch, steps, _ = emissions.shape
        safe_tags = tags.masked_fill(tags.eq(PAD_TAG), 0)
        gold = self.start[safe_tags[:, 0]] + emissions[:, 0].gather(1, safe_tags[:, 0:1]).squeeze(1)
        for t in range(1, steps):
            active = (lengths > t)
            step_score = self.transitions[safe_tags[:, t-1], safe_tags[:, t]] + emissions[:, t].gather(1, safe_tags[:, t:t+1]).squeeze(1)
            gold = gold + step_score * active
        last_tags = safe_tags[torch.arange(batch, device=tags.device), lengths - 1]
        gold = gold + self.end[last_tags]
        alpha = self.start + emissions[:, 0]
        for t in range(1, steps):
            scores = alpha.unsqueeze(2) + self.transitions.unsqueeze(0) + emissions[:, t].unsqueeze(1)
            next_alpha = torch.logsumexp(scores, dim=1)
            alpha = torch.where((lengths > t).unsqueeze(1), next_alpha, alpha)
        partition = torch.logsumexp(alpha + self.end, dim=1)
        return (partition - gold).mean()

    def decode(self, emissions, lengths):
        paths = []
        for i, length in enumerate(lengths.tolist()):
            score = self.start + emissions[i, 0]
            backpointers = []
            for t in range(1, length):
                candidates = score.unsqueeze(1) + self.transitions
                best_score, best_prev = candidates.max(dim=0)
                score = best_score + emissions[i, t]
                backpointers.append(best_prev)
            tag = int((score + self.end).argmax())
            path = [tag]
            for pointer in reversed(backpointers):
                tag = int(pointer[tag]); path.append(tag)
            paths.append(list(reversed(path)))
        return paths

class NERModel(nn.Module):
    def __init__(self, use_char, use_crf):
        super().__init__()
        self.use_char, self.use_crf = use_char, use_crf
        self.word_embedding = nn.Embedding(len(word2id), WORD_DIM, padding_idx=word2id[PAD_WORD])
        input_dim = WORD_DIM
        if use_char:
            self.char_embedding = nn.Embedding(len(char2id), 30, padding_idx=char2id[PAD_CHAR])
            self.char_conv = nn.Conv1d(30, 50, kernel_size=3, padding=1)
            input_dim += 50
        self.encoder = nn.LSTM(input_dim, HIDDEN_DIM, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(0.2)
        self.classifier = nn.Linear(HIDDEN_DIM * 2, len(tag2id))
        self.crf = LinearChainCRF(len(tag2id)) if use_crf else None

    def forward(self, words, chars, char_lengths, lengths):
        features = [self.word_embedding(words)]
        if self.use_char:
            b, t, c = chars.shape
            embedded = self.char_embedding(chars.reshape(b*t, c)).transpose(1, 2)
            convolved = torch.relu(self.char_conv(embedded)).transpose(1, 2)
            positions = torch.arange(c, device=chars.device).unsqueeze(0)
            mask = positions < char_lengths.reshape(-1, 1)
            convolved = convolved.masked_fill(~mask.unsqueeze(-1), torch.finfo(convolved.dtype).min)
            char_vectors = convolved.max(dim=1).values.reshape(b, t, -1)
            features.append(char_vectors)
        packed = nn.utils.rnn.pack_padded_sequence(torch.cat(features, dim=-1), lengths.cpu(), batch_first=True, enforce_sorted=False)
        encoded, _ = self.encoder(packed)
        encoded, _ = nn.utils.rnn.pad_packed_sequence(encoded, batch_first=True, total_length=words.shape[1])
        return self.classifier(self.dropout(encoded))

    def loss(self, emissions, tags, lengths):
        token_loss = nn.functional.cross_entropy(emissions.reshape(-1, emissions.shape[-1]), tags.reshape(-1),
                                                 weight=tag_weights.to(emissions.device), ignore_index=PAD_TAG)
        if self.use_crf: return self.crf.nll(emissions, tags, lengths) + 0.5 * token_loss
        return token_loss

    def predict(self, emissions, lengths):
        if self.use_crf: return self.crf.decode(emissions, lengths)
        return [emissions[i, :n].argmax(-1).tolist() for i, n in enumerate(lengths.tolist())]

@torch.no_grad()
def evaluate_loss(model, loader):
    model.eval(); total, count = 0.0, 0
    for words, chars, char_lengths, tags, lengths in loader:
        words, chars, char_lengths, tags, lengths = [x.to(DEVICE) for x in (words, chars, char_lengths, tags, lengths)]
        loss = model.loss(model(words, chars, char_lengths, lengths), tags, lengths)
        total += float(loss) * len(lengths); count += len(lengths)
    return total / max(count, 1)

@torch.no_grad()
def evaluate_entity_f1(model):
    model.eval(); tp = fp = fn = 0
    for batch in valid_loader:
        words, chars, char_lengths, tags, lengths = [x.to(DEVICE) for x in batch]
        paths = model.predict(model(words, chars, char_lengths, lengths), lengths)
        for i, path in enumerate(paths):
            n = int(lengths[i])
            gold_tags = [id2tag[int(x)] for x in tags[i, :n].tolist()]
            pred_tags = [id2tag[x] for x in path]
            gold_spans, pred_spans = set(parse_spans(gold_tags)), set(parse_spans(pred_tags))
            tp += len(gold_spans & pred_spans); fp += len(pred_spans - gold_spans); fn += len(gold_spans - pred_spans)
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    return 2 * precision * recall / (precision + recall) if precision + recall else 0.0

def train_model(name, use_char, use_crf):
    model = NERModel(use_char, use_crf).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    best_f1, best_loss, best_state, stale, history = -1.0, float('inf'), None, 0, []
    start_time = time.time()
    for epoch in range(1, EPOCHS + 1):
        model.train(); total, count = 0.0, 0
        for words, chars, char_lengths, tags, lengths in train_loader:
            words, chars, char_lengths, tags, lengths = [x.to(DEVICE) for x in (words, chars, char_lengths, tags, lengths)]
            optimizer.zero_grad(set_to_none=True)
            loss = model.loss(model(words, chars, char_lengths, lengths), tags, lengths)
            loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 5.0); optimizer.step()
            total += float(loss.detach()) * len(lengths); count += len(lengths)
        valid_loss = evaluate_loss(model, valid_loader)
        valid_f1 = evaluate_entity_f1(model)
        history.append({'epoch': epoch, 'train_loss': total/max(count,1), 'valid_loss': valid_loss, 'valid_entity_f1': valid_f1})
        print(f'{name}: epoch {epoch}/{EPOCHS} train_loss={total/max(count,1):.4f} valid_loss={valid_loss:.4f} valid_entity_f1={valid_f1:.4f}')
        if valid_f1 > best_f1 or (valid_f1 == best_f1 and valid_loss < best_loss):
            best_f1, best_loss, stale = valid_f1, valid_loss, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            stale += 1
            if stale >= PATIENCE: break
    if best_state is None: raise RuntimeError(f'{name} training produced no checkpoint')
    model.load_state_dict(best_state)
    print(f'{name}: restored best validation checkpoint; elapsed={time.time()-start_time:.1f}s')
    return model, history

print('Model components initialized successfully.')

Model components initialized successfully.


## 5. Huấn luyện ba mô hình và đánh giá

Dùng cùng split, seed, word vocabulary, optimizer và tiêu chí chọn checkpoint (validation loss). Mọi embedding/encoder được khởi tạo ngẫu nhiên.

In [6]:
configs = {
    'Baseline 1: Word-BiLSTM-Softmax': (False, False),
    'Baseline 2 / Ablation: Word-BiLSTM-CRF': (False, True),
    'Proposed: Word+CharCNN-BiLSTM-CRF': (True, True),
}
models, histories = {}, {}
for name, (use_char, use_crf) in configs.items():
    models[name], histories[name] = train_model(name, use_char, use_crf)

@torch.no_grad()
def predict_rows(model, rows):
    model.eval(); predictions = []
    for batch in make_loader(rows):
        words, chars, char_lengths, _, lengths = [x.to(DEVICE) for x in batch]
        emissions = model(words, chars, char_lengths, lengths)
        predictions.extend([[id2tag[x] for x in path] for path in model.predict(emissions, lengths)])
    return predictions

predictions = {name: predict_rows(model, test) for name, model in models.items()}
assert all(len(p) == len(test) for p in predictions.values())
print('Prediction complete:', {name: len(p) for name, p in predictions.items()})

Baseline 1: Word-BiLSTM-Softmax: epoch 1/12 train_loss=1.8531 valid_loss=2.0475 valid_entity_f1=0.0000
Baseline 1: Word-BiLSTM-Softmax: epoch 2/12 train_loss=1.5166 valid_loss=2.1268 valid_entity_f1=0.0000
Baseline 1: Word-BiLSTM-Softmax: epoch 3/12 train_loss=1.3569 valid_loss=2.3301 valid_entity_f1=0.0000
Baseline 1: Word-BiLSTM-Softmax: epoch 4/12 train_loss=1.1596 valid_loss=2.5669 valid_entity_f1=0.0065
Baseline 1: Word-BiLSTM-Softmax: epoch 5/12 train_loss=0.9606 valid_loss=2.8556 valid_entity_f1=0.0143
Baseline 1: Word-BiLSTM-Softmax: epoch 6/12 train_loss=0.7736 valid_loss=3.0181 valid_entity_f1=0.0231
Baseline 1: Word-BiLSTM-Softmax: epoch 7/12 train_loss=0.5924 valid_loss=3.5551 valid_entity_f1=0.0218
Baseline 1: Word-BiLSTM-Softmax: epoch 8/12 train_loss=0.4609 valid_loss=3.8624 valid_entity_f1=0.0256
Baseline 1: Word-BiLSTM-Softmax: epoch 9/12 train_loss=0.3506 valid_loss=4.2942 valid_entity_f1=0.0258
Baseline 1: Word-BiLSTM-Softmax: epoch 10/12 train_loss=0.2664 valid_loss

## 6. Bằng chứng thực nghiệm: exact-span F1 theo độ phổ biến

Một span đúng khi khớp chính xác start, end và loại entity. Precision/recall/F1 tính micro. Theo nhóm, gold được phân vào nhóm theo tần suất surface form train; predicted span được phân nhóm tương tự để tính FP nhất quán. Support là số gold span trong nhóm.

In [7]:
GROUPS = ['all', 'unseen', 'rare (1-2)', 'frequent (3+)']
def sentence_counts(tokens, gold_tags, pred_tags):
    gold = parse_spans(gold_tags); pred = parse_spans(pred_tags)
    gold_by_group = collections.defaultdict(set); pred_by_group = collections.defaultdict(set)
    for span in gold: gold_by_group[entity_group(tokens, span)].add(span)
    for span in pred: pred_by_group[entity_group(tokens, span)].add(span)
    result = {}
    for group in GROUPS[1:]:
        g, p = gold_by_group[group], pred_by_group[group]
        result[group] = (len(g & p), len(p - g), len(g - p), len(g))
    g_all, p_all = set(gold), set(pred)
    result['all'] = (len(g_all & p_all), len(p_all - g_all), len(g_all - p_all), len(g_all))
    return result

def aggregate_metrics(model_predictions):
    totals = {g: np.zeros(4, dtype=np.int64) for g in GROUPS}
    per_sentence = []
    for row, pred_tags in zip(test, model_predictions):
        counts = sentence_counts(row['tokens'], row['tags'], pred_tags)
        per_sentence.append(counts)
        for group, values in counts.items(): totals[group] += values
    metrics = {}
    for group, (tp, fp, fn, support) in totals.items():
        precision = tp / (tp + fp) if tp + fp else 0.0
        recall = tp / (tp + fn) if tp + fn else 0.0
        f1 = 2*precision*recall/(precision+recall) if precision+recall else 0.0
        metrics[group] = {'precision': precision, 'recall': recall, 'f1': f1, 'support': int(support), 'tp': int(tp), 'fp': int(fp), 'fn': int(fn)}
    return metrics, per_sentence

all_metrics, sentence_level = {}, {}
for name, pred in predictions.items(): all_metrics[name], sentence_level[name] = aggregate_metrics(pred)
score_rows = []
for name, group_metrics in all_metrics.items():
    for group in GROUPS:
        m = group_metrics[group]
        score_rows.append({'model': name, 'entity_group': group, 'precision': m['precision'], 'recall': m['recall'], 'F1': m['f1'], 'support': m['support']})
results = pd.DataFrame(score_rows)
display(results.round(4))
results.to_csv('wnut17_entity_metrics.csv', index=False)

                                 model  entity_group  precision  recall     F1  support
       Baseline 1: Word-BiLSTM-Softmax           all     0.0548  0.0111 0.0185     1079
       Baseline 1: Word-BiLSTM-Softmax        unseen     0.0563  0.0114 0.0189     1055
       Baseline 1: Word-BiLSTM-Softmax    rare (1-2)     0.0000  0.0000 0.0000       22
       Baseline 1: Word-BiLSTM-Softmax frequent (3+)     0.0000  0.0000 0.0000        2
Baseline 2 / Ablation: Word-BiLSTM-CRF           all     0.1174  0.0287 0.0462     1079
Baseline 2 / Ablation: Word-BiLSTM-CRF        unseen     0.1115  0.0275 0.0441     1055
Baseline 2 / Ablation: Word-BiLSTM-CRF    rare (1-2)     0.5000  0.0909 0.1538       22
Baseline 2 / Ablation: Word-BiLSTM-CRF frequent (3+)     0.0000  0.0000 0.0000        2
     Proposed: Word+CharCNN-BiLSTM-CRF           all     0.0813  0.1121 0.0943     1079
     Proposed: Word+CharCNN-BiLSTM-CRF        unseen     0.0781  0.1090 0.0910     1055
     Proposed: Word+CharCNN-BiLS

## 7. Ablation và bootstrap CI cho chênh lệch F1

Ablation chính là mô hình word-only CRF (tắt character; cùng word BiLSTM và CRF với proposed). Bootstrap resample câu có hoàn lại, tính lại micro-F1 mỗi lần; CI percentile 95% cho `F1(character) - F1(word-only CRF)`. CI không bao gồm biến thiên do huấn luyện lại.

In [8]:
def f1_from_counts(counts):
    tp, fp, fn = counts[:3]
    p = tp/(tp+fp) if tp+fp else 0.0
    r = tp/(tp+fn) if tp+fn else 0.0
    return 2*p*r/(p+r) if p+r else 0.0

def bootstrap_delta(group, n_boot=1000, seed=SEED):
    rng = np.random.default_rng(seed)
    a_name, b_name = 'Proposed: Word+CharCNN-BiLSTM-CRF', 'Baseline 2 / Ablation: Word-BiLSTM-CRF'
    a, b = sentence_level[a_name], sentence_level[b_name]
    deltas = np.empty(n_boot)
    for k in range(n_boot):
        indices = rng.integers(0, len(test), len(test))
        totals = []
        for per_model in (a, b):
            summed = np.zeros(4, dtype=np.int64)
            for i in indices: summed += np.asarray(per_model[i][group])
            totals.append(f1_from_counts(summed))
        deltas[k] = totals[0] - totals[1]
    point = all_metrics[a_name][group]['f1'] - all_metrics[b_name][group]['f1']
    low, high = np.quantile(deltas, [0.025, 0.975])
    return point, float(low), float(high), float(np.mean(deltas > 0))

ablation_rows = []
for group in GROUPS:
    delta, low, high, prob_positive = bootstrap_delta(group)
    ablation_rows.append({'entity_group': group, 'F1_word+char': all_metrics['Proposed: Word+CharCNN-BiLSTM-CRF'][group]['f1'],
                          'F1_word_only_CRF': all_metrics['Baseline 2 / Ablation: Word-BiLSTM-CRF'][group]['f1'],
                          'delta_F1': delta, 'bootstrap_95%_CI': f'[{low:.4f}, {high:.4f}]',
                          'bootstrap_P(delta>0)': prob_positive})
ablation_results = pd.DataFrame(ablation_rows)
display(ablation_results.round(4))
ablation_results.to_csv('wnut17_ablation_bootstrap.csv', index=False)

plot_df = results[results.entity_group.isin(['unseen', 'rare (1-2)', 'frequent (3+)'])]
ax = plot_df.pivot(index='entity_group', columns='model', values='F1').plot(kind='bar', figsize=(10, 5))
ax.set_ylabel('Entity exact-span micro-F1'); ax.set_xlabel('Test gold entity group'); ax.set_ylim(0, 1)
plt.xticks(rotation=0); plt.tight_layout(); plt.show()

 entity_group  F1_word+char  F1_word_only_CRF  delta_F1  bootstrap_95%_CI  bootstrap_P(delta>0)
          all        0.0943            0.0462    0.0481  [0.0277, 0.0662]                 1.000
       unseen        0.0910            0.0441    0.0469  [0.0265, 0.0655]                 1.000
   rare (1-2)        0.3243            0.1538    0.1705 [-0.0143, 0.3846]                 0.964
frequent (3+)        0.0000            0.0000    0.0000  [0.0000, 0.0000]                 0.000
wnut17_character_ner.ipynb:cell-8:38: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


## 8. Phân tích lỗi và kết luận dựa trên số liệu

Liệt kê lỗi span (bỏ sót, gán sai ranh giới/loại, dự đoán thừa) trên một số câu test; đồng thời tổng hợp FP/FN theo nhóm unseen/rare/frequent. Kết luận cuối cùng được sinh từ metric thực tế, không giả định character luôn có lợi.

In [9]:
def span_text(tokens, span): return f"{' '.join(tokens[span[0]:span[1]])} [{span[2]}]"
def collect_errors(pred_tags):
    errors = []
    for sent_id, (row, predicted) in enumerate(zip(test, pred_tags)):
        gold_spans, pred_spans = set(parse_spans(row['tags'])), set(parse_spans(predicted))
        missed, extra = gold_spans - pred_spans, pred_spans - gold_spans
        if not missed and not extra: continue
        for span in sorted(missed):
            errors.append({'sentence_id': sent_id, 'error': 'FN/missed or boundary/type mismatch', 'entity': span_text(row['tokens'], span),
                           'group': entity_group(row['tokens'], span), 'context': ' '.join(row['tokens'])[:220]})
        for span in sorted(extra):
            errors.append({'sentence_id': sent_id, 'error': 'FP/spurious or boundary/type mismatch', 'entity': span_text(row['tokens'], span),
                           'group': entity_group(row['tokens'], span), 'context': ' '.join(row['tokens'])[:220]})
    return errors

proposed_name = 'Proposed: Word+CharCNN-BiLSTM-CRF'
error_rows = collect_errors(predictions[proposed_name])
error_df = pd.DataFrame(error_rows)
if len(error_df):
    display(error_df.head(20))
    display(error_df.groupby(['group', 'error']).size().rename('count').reset_index())
    error_df.to_csv('wnut17_error_examples.csv', index=False)
else:
    print('Không có lỗi span trên test — hãy kiểm tra kích thước test và parser nhãn.')

char = all_metrics[proposed_name]
word = all_metrics['Baseline 2 / Ablation: Word-BiLSTM-CRF']
for group in ['unseen', 'rare (1-2)', 'all']:
    delta, low, high, _ = bootstrap_delta(group)
    direction = 'tăng' if delta > 0 else ('giảm' if delta < 0 else 'không đổi')
    evidence = 'CI không chứa 0' if low > 0 or high < 0 else 'CI chứa 0, chưa có bằng chứng chắc chắn về khác biệt'
    print(f"{group}: Word+Char F1={char[group]['f1']:.4f} (n={char[group]['support']}), "
          f"word-only CRF F1={word[group]['f1']:.4f}; Δ={delta:+.4f}, 95% CI [{low:+.4f}, {high:+.4f}] -> {direction}; {evidence}.")

print('\nKết luận: Character representation có giúp unseen/rare hay không phải dựa vào Δ-F1 và bootstrap CI ngay phía trên. '
      'CI chứa 0 nghĩa là thí nghiệm này chưa đủ bằng chứng để kết luận cải thiện; không diễn giải là hai mô hình tương đương. '
      'Xem các FN/FP và nhóm lỗi để nhận diện lỗi do viết biến thể/OOV, ranh giới entity, nhãn mơ hồ hoặc entity không đủ ngữ cảnh. '
      'Hạn chế: một seed và một split; bootstrap chỉ lượng hóa biến thiên lấy mẫu test, và surface-form matching có thể đánh đồng entity khác ngữ nghĩa.')
print('Saved: wnut17_entity_metrics.csv, wnut17_ablation_bootstrap.csv, wnut17_error_examples.csv')

 sentence_id                                 error                                          entity  group                                                                                                                                                                                      context
           0 FP/spurious or boundary/type mismatch                   military spokesman [location] unseen                                                    & gt ; * The soldier was killed when another avalanche hit an army barracks in the northern area of Sonmarg , said a military spokesman .
           4   FN/missed or boundary/type mismatch                  Avalanche Rescue Teams [group] unseen     The bodies of the soldiers were recovered after the concerted efforts of the Avalanche Rescue Teams ( ART ) , which is equipped to work in inhospitable terrain and weather conditions .
           4   FN/missed or boundary/type mismatch                                     ART [group] unseen     The b